<a href="https://colab.research.google.com/github/faizanarif2/PyTorch_NeuralNetworks/blob/main/CNN_SyntheticLineDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader,Dataset
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint,EarlyStopping

In [2]:
class SyntheticLineDataset(Dataset):
#Made custom Dataset usinng MNIST
#Made synthetic line images
    def __init__(self, base_dataset, seq_len=4):
        self.base_dataset = base_dataset
        self.seq_len = seq_len

    def __len__(self):
        return len(self.base_dataset) // self.seq_len

    def __getitem__(self, idx):
        images, labels = [], []
        for i in range(self.seq_len):
            img, label = self.base_dataset[idx * self.seq_len + i]
            images.append(img)
            labels.append(label)

        line_image = torch.cat(images, dim=2)
        line_labels = torch.tensor(labels, dtype=torch.long)
        return line_image, line_labels

In [3]:
class DataModule(pl.LightningDataModule):

  def __init__(self,data_dir="./data",batch_size: int=32,seq_len: int=4):

    super().__init__()
    self.data_dir=data_dir
    self.batch_size=batch_size
    self.seq_len=seq_len
    self.transform=transforms.ToTensor()

  def prepare_data(self):
    MNIST(self.data_dir,train=True,download=True)
    MNIST(self.data_dir,train=False,download=True)

  def setup(self,stage=None):
    raw_train=MNIST(self.data_dir,train=True,transform=self.transform)
    raw_val=MNIST(self.data_dir,train=False,transform=self.transform)

    self.train_dataset=SyntheticLineDataset(raw_train,seq_len=self.seq_len)
    self.val_dataset=SyntheticLineDataset(raw_val,seq_len=self.seq_len)

  def train_dataloader(self):
    return DataLoader(self.train_dataset,batch_size=self.batch_size,shuffle=True)

  def val_dataloader(self):
    return DataLoader(self.val_dataset,batch_size=self.batch_size)






In [4]:
from torch.nn.modules.dropout import Dropout
from torch.nn.modules.linear import Linear
from torch.nn.modules.flatten import Flatten
from torch.nn.modules.container import Sequential
from torch.nn.modules.pooling import AdaptiveAvgPool2d
from torch.nn.modules.activation import ReLU
class ModelCNN(pl.LightningModule):

  def __init__(self,seq_len: int=4,num_classes:int=10,lr:float=0.001):

    super().__init__()
    self.save_hyperparameters()
    self.seq_len=seq_len
    self.num_classes=num_classes

    self.conv=nn.Sequential(
        nn.Conv2d(1,32,kernel_size=3,padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d((1,7*seq_len))

    )

    self.linearLayers=nn.Sequential(
        nn.Flatten(),
        nn.Linear(128*7*seq_len,256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256,seq_len*num_classes)
    )

    self.loss_fn=nn.CrossEntropyLoss()

  def forward(self,x):
    features=self.conv(x)
    logits=self.linearLayers(features)
    return logits.view(-1,self.seq_len,self.num_classes)

  def training_step(self,batch,batch_idx):
    image,labels=batch
    predictions=self(image)
    loss=self.loss_fn(predictions.view(-1,self.num_classes),labels.view(-1))
    self.log("train_loss",loss,prog_bar=True,on_epoch=True)
    return loss


  def validation_step(self,batch,batch_idx):
    image,labels=batch
    predictions=self(image)
    val_loss=self.loss_fn(predictions.view(-1,self.num_classes),labels.view(-1))

    prediction_labels=predictions.argmax(dim=-1)
    char_acc=(prediction_labels==labels).float().mean()


    self.log("val_loss",val_loss,prog_bar=True,on_epoch=True)
    self.log("val_char_acc",char_acc,prog_bar=True)
    return val_loss

  def configure_optimizers(self):
    return torch.optim.Adam(self.parameters(),lr=self.hparams.lr)











In [5]:
seq_len=4
datamodule=DataModule(seq_len=seq_len)
model=ModelCNN(seq_len=seq_len)

checkpoint_callback=ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="line-model"
)


early_stop_callback=EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min"
)

trainer=pl.Trainer(
    max_epochs=5,
    callbacks=[checkpoint_callback,early_stop_callback],
    accelerator="auto"
)

trainer.fit(model,datamodule=datamodule)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ conv         │ Sequential       │ 93.1 K │ train │     0 │
│ 1 │ linearLayers │ Sequential       │  928 K │ train │     0 │
│ 2 │ loss_fn      │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 1.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.0 M                                                                                                
Total estimated model params size (MB): 4.085                                                                      
Modules in train mode: 20                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.
